# Gemma-Titans Hybrid: TPU v5e Online Learning Prototype

This notebook implements a hybrid architecture combining **Gemma-2B** with **Titans Neural Long-Term Memory (NLTM)**. 

**Architecture Key Features:**
- **Parallel Memory:** Titans memory runs alongside standard Attention.
- **Learned Gating:** A scalar gate balances short-term context and long-term memory.
- **Modular Weights:** We only train and save the "Titans Delta" (~50MB), keeping the 5GB Gemma weights frozen.

## 1. Environment Setup (TPU v5e-1)
Run the cells below to install the JAX/Flax stack and the official Gemma framework.

In [ ]:
# 1. Install specific versions of shared dependencies first
!pip install "ml-dtypes==0.4.0" "protobuf==5.28.3"

# 2. Install JAX for TPU
!pip install "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

# 3. Install TensorFlow (Required for Kauldron/Gemma internal type checking on TPU)
!pip install tensorflow==2.18.0 tensorflow-tpu==2.18.0 --find-links=https://storage.googleapis.com/libtpu-tf-releases/index.html

# 4. Install project dependencies with critical version locks
!pip install typeguard==4.4.1 gemma==3.3.0 kauldron==1.3.0 flax==0.12.5 optax==0.2.6
!pip install einops einx treescope jaxtyping sentencepiece datasets

In [ ]:
# Clone the repository
!git clone https://github.com/andrew-veriga/Titans_jax.git
%cd Titans_jax

## 2. Initialization & Model Surgery
We initialize the hybrid structure and extract the initial "Titans Delta" weights.

In [ ]:
import jax
import jax.numpy as jnp
from kauldron import typing as ktyping # Add this
ktyping.config.enable_type_checks(False) # Disable problematic type checks

from gemma.gm.nn import _config, _modules
from gemma_titans import GemmaTitansTransformer
from model_loader import stitch_hybrid_model, load_titans_delta
import os

# 1. Define Gemma-2B Config
config_2b = _config.TransformerConfig(
    num_embed=256000,
    embed_dim=2048,
    hidden_dim=16384,
    num_heads=8,
    head_dim=256,
    num_kv_heads=1,
    final_logit_softcap=30.0,
    use_post_attn_norm=False,
    use_post_ffw_norm=False,
    attention_types=[_modules.AttentionType.GLOBAL] * 18, # 18 layers for 2B
)

model = GemmaTitansTransformer(config=config_2b, dtype=jnp.bfloat16)

print("Initializing Hybrid Structure...")
rng = jax.random.PRNGKey(0)
dummy_tokens = jnp.ones((1, 1), dtype=jnp.int32)
variables = model.init(rng, tokens=dummy_tokens)
params = variables['params']

# In a real scenario, you would now load the official Gemma weights from HuggingFace 
# and use stitch_hybrid_model to merge them with these initialized memory weights.

## 3. Dataset Preparation
We use a subset of OpenAssistant for dialogue training.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("OpenAssistant/oasst1", split="train[:100]")
print(f"Loaded {len(dataset)} dialogue samples.")

## 4. Phase 1: Continued Pre-Training (CPT)
We freeze the Gemma weights and train only the `memory` and `memory_gate` parameters.

In [ ]:
import optax

# Define the mask: Only train Titans parameters
def is_trainable(path, val):
    path_str = "/".join([str(p.name) for p in path])
    return 'memory' in path_str or 'memory_gate' in path_str

mask = jax.tree_util.tree_map_with_path(is_trainable, params)

# Setup Optimizer with masking
tx = optax.chain(
    optax.masked(optax.adam(learning_rate=1e-4), mask)
)
opt_state = tx.init(params)

@jax.jit
def train_step(params, opt_state, tokens):
    def loss_fn(p):
        logits = model.apply({'params': p}, tokens=tokens).logits
        # Simple Cross-Entropy Loss implementation would go here
        return jnp.mean(logits**2) # Dummy loss for prototype demo
    
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = tx.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

print("Starting training step...")
params, opt_state, loss = train_step(params, opt_state, dummy_tokens)
print(f"Initial loss: {loss}")

## 5. Dialogue Inference with Dynamic Memory
This demonstrates how the model updates its internal state (`TitansLayerCache`) during a conversation.

In [ ]:
def dialogue_turn(params, cache, user_input_tokens):
    # 1. Forward pass returns updated cache (including updated Titans memory)
    output = model.apply({'params': params}, tokens=user_input_tokens, cache=cache)
    logits = output.logits
    new_cache = output.cache
    
    # 2. Logic to extract the response token would go here
    return logits, new_cache

# Initial state
cache = model.init_cache(batch_size=1, dtype=jnp.bfloat16, cache_length=512)

print("Simulation: User enters a turn...")
user_tokens = jnp.array([[1, 2, 3, 4]])
logits, cache = dialogue_turn(params, cache, user_tokens)
print("Memory state updated automatically in cache.")

## 6. Save/Load Titans Delta
We save ONLY the trained memory weights to keep the file size minimal.

In [ ]:
import orbax.checkpoint as ocp

def save_titans_only(params, path):
    # Filter to save only Titans parameters
    titans_delta = jax.tree_util.tree_map_with_path(
        lambda path, val: val if is_trainable(path, val) else None, 
        params
    )
    
    checkpointer = ocp.StandardCheckpointer()
    checkpointer.save(os.path.abspath(path), titans_delta, force=True)
    print(f"Saved Titans Delta to {path}")

save_titans_only(params, "./titans_memory_trained")